# Team Member Contribution Table

| Member | Contribution |
| ------ | ------------ |
| Petter (SM) | See team clustering work summarized in this notebook. |
| Kim | See hierarchical clustering section below. |
| Elliot | See team clustering work summarized in this notebook. |
| Agust | See team clustering work summarized in this notebook. |

# A7 Clustering and PCA of AIMO Observations

This notebook combines the results from the hierarchical-clustering report in `reporting/A7_h.ipynb` and the K-means results documented in `ml/models/research/A7/A7_cluster_kmeans.ipynb`. The goal of Lab 7 was to:

- perform clustering of AIMO observations while excluding score columns from the clustering inputs
- interpret the resulting clusters
- perform PCA of the observations
- plot the first two principal component scores in a 2D scatterplot coloured by cluster
- interpret the PCA plot

Two clustering approaches were used in the sprint:

- **K-means clustering** on a 40-feature deviation representation of AIMO observations
- **Hierarchical clustering** on a 14-feature movement-quality representation of AIMO observations

Because the two notebooks use different processed versions of the AIMO data, the results should be read as **complementary views** of the same domain rather than as a one-to-one comparison of identical inputs.

## Data representations used

The sprint produced two clustering analyses on two processed AIMO representations:

| Method | Source file | Observations | Features used for clustering | Excluded columns |
| ------ | ----------- | ------------ | ---------------------------- | ---------------- |
| K-means | `ml/models/data/processed/AimoScore_WeakLink_big_scores.csv` | 2094 | 40 biomechanical deviation features | `AimoScore`, `EstimatedScore`, `ID` |
| Hierarchical | `ml/models/data/processed/20190108_scores_and_weak_links.csv` | 2096 | 14 movement-quality variables | `ID`, `Date`, `SCORE` |

In both cases, the score columns were excluded from the clustering itself. They were only used afterwards for interpretation and validation.

## K-means clustering and PCA

The K-means analysis used a larger, more detailed feature representation with 40 biomechanical deviation variables. Missing values were imputed with the median, then all features were standardized so that Euclidean distance in K-means would not be dominated by variables with larger numeric scale.

The number of clusters was chosen using the **elbow method**. The within-cluster sum of squares (inertia) was plotted for `K = 1` to `10`, and the curve showed a clear bend at **K = 3**. The final K-means model was then fit with `n_init = 20` and `random_state = 42`.

### K-means cluster results

| Cluster | Size | Share | Mean AimoScore | Interpretation |
| ------- | ---- | ----- | -------------- | -------------- |
| 0 | 1098 | 52% | 0.703 | Good movers with low deviations across the feature set |
| 1 | 451 | 22% | 0.418 | Poor movers with a timing-dominated failure pattern |
| 2 | 545 | 26% | 0.438 | Poor movers with angle-dominated deviation and high time deviation |

The K-means result is interesting because the scores were not used during clustering, yet the recovered groups align strongly with movement quality. Cluster 0 is clearly the strongest group, with much higher mean `AimoScore` than the other two clusters. This suggests that the biomechanical deviation features alone carry a strong quality signal.

The two weaker clusters have similar average `AimoScore`, but they represent different error profiles:

- **Cluster 1** is dominated by **timing-related deviations**, especially `No_1_Time_Deviation` and `No_2_Time_Deviation`, together with some specific angle outliers.
- **Cluster 2** shows broader elevation across angle-deviation features such as `No_1`, `No_2`, `No_4`, and `No_6`, while also having the highest time deviations. This makes it a more globally deviant movement pattern.

The strongest discriminating features were reported to be mainly NASM-based variables (`No_12_NASM`, `No_11_NASM`, `No_5_NASM`) followed by angle-deviation variables (`No_13_Angle`, `No_10_Angle`).

### K-means PCA interpretation

For the K-means notebook, PCA was computed on the same scaled 40-feature matrix. The first two principal components explained:

| Component | Explained variance |
| --------- | ------------------ |
| PC1 | 18.6% |
| PC2 | 12.5% |
| Total | 31.1% |

The PCA interpretation in the K-means notebook was:

- **PC1** represents overall deviation severity. Points further to the right have larger deviations across the feature set, while points on the left are closer to the reference movement pattern.
- **PC2** separates different *types* of poor movement, especially the contrast between timing-dominated and angle-dominated failure modes.

This means the PCA plot supports the 3-cluster K-means result well: one cluster represents better movement quality, while the other two represent different kinds of poorer movement rather than one single undifferentiated failure group.

## Hierarchical clustering and PCA

The hierarchical clustering analysis used a smaller and more directly interpretable feature representation with 14 movement-quality variables such as forward head, arm fall forward, shoulder elevation, knee movement, and heel rise. The data was standardized before clustering.

Agglomerative hierarchical clustering with **Ward linkage** was applied. A dendrogram was produced and several cuts of the hierarchy were evaluated. To support the final choice of cluster count, silhouette scores were computed for `k = 2` to `7`.

### Hierarchical cluster selection

| k | Silhouette score | Smallest cluster | Largest cluster |
| - | ---------------- | ---------------- | --------------- |
| 2 | 0.2634 | 322 | 1774 |
| 7 | 0.2385 | 77 | 820 |
| 6 | 0.2228 | 86 | 820 |
| 5 | 0.1932 | 86 | 820 |
| 4 | 0.1726 | 322 | 820 |
| 3 | 0.1629 | 322 | 954 |

The highest silhouette score was obtained for **k = 2**, so the hierarchical result was reported as a two-cluster solution. The value is moderate rather than high, which suggests that the result should be interpreted as a broad but still meaningful split.

### Hierarchical cluster results

| Cluster | Size | Interpretation |
| ------- | ---- | -------------- |
| 0 | 1774 | Large general cluster with lower outward-knee and heel-rise values |
| 1 | 322 | Smaller distinct subgroup with strong outward-knee movement and heel rise |

A subset of the most informative mean values is shown below:

| Cluster | ForwardHead | LeftArmFallForward | RightArmFallForward | RightKneeMovesOutward | LeftHeelRises | RightHeelRises |
| ------- | ----------- | ------------------ | ------------------- | --------------------- | ------------- | -------------- |
| 0 | 0.350 | 0.984 | 0.983 | 0.033 | 0.012 | 0.008 |
| 1 | 0.034 | 1.067 | 0.318 | 1.371 | 0.317 | 0.625 |

The most important standardized deviations from the dataset mean were:

- **Cluster 0:** lower `RightKneeMovesOutward`, lower `RightHeelRises`, lower `LeftHeelRises`, slightly higher `RightArmFallForward`
- **Cluster 1:** much higher `RightKneeMovesOutward`, higher `RightHeelRises`, higher `LeftHeelRises`, lower `RightArmFallForward`

This means the hierarchical method isolated a specific subgroup whose movement pattern is strongly associated with **heel rise and outward knee movement**, especially on the right side.

### Hierarchical PCA interpretation

The PCA of the hierarchical-clustering feature matrix explained:

| Component | Explained variance |
| --------- | ------------------ |
| PC1 | 21.68% |
| PC2 | 14.77% |
| Total | 36.45% |

The dominant loadings showed that:

- **PC1** is mainly driven by `RightHeelRises`, `RightKneeMovesOutward`, and `LeftHeelRises` in the positive direction, and by `RightArmFallForward` and `ForwardHead` in the negative direction
- **PC2** is mainly driven by `LeftArmFallForward` and `RightArmFallForward` in the positive direction, and by `RightShoulderElevation` and `LeftShoulderElevation` in the negative direction

The cluster centroids differed much more on **PC1** than on **PC2** (`-0.580` vs `3.193` on PC1, but only `0.060` vs `-0.333` on PC2). This means the two hierarchical clusters are separated mainly along the first component. In practical terms, the PCA plot supports the interpretation that the cluster difference is primarily about **heel rise and outward-knee behavior**, not about shoulder variation.

## Comparison of the two clustering approaches

| Aspect | K-means | Hierarchical |
| ------ | ------- | ------------ |
| Input representation | 40 deviation features | 14 movement-quality variables |
| Number of observations | 2094 | 2096 |
| How number of clusters was chosen | Elbow method | Dendrogram plus silhouette score |
| Final number of clusters | 3 | 2 |
| Main result | One good-movement cluster and two different poor-movement clusters | One broad main cluster and one distinct lower-body compensation subgroup |
| PCA variance explained by PC1+PC2 | 31.1% | 36.45% |

The two methods tell a consistent story, but at different levels of detail.

The **K-means result** is more granular. It identifies a strong "good movement" cluster and then separates the weaker observations into two different failure modes. This is useful when the goal is to distinguish several categories of biomechanical behaviour.

The **hierarchical result** is simpler and more directly interpretable. Instead of creating multiple poor-movement categories, it isolates one clear subgroup with a strong lower-body compensation pattern and leaves the rest of the observations in a broad main cluster. This is useful when the goal is interpretability and visual explanation through the dendrogram.

Taken together, the two analyses suggest that the AIMO observations contain both:

- a broad quality axis, where some subjects are clearly closer to the reference movement pattern than others
- additional substructure inside the poorer observations, where different compensation or failure modes can appear

## Final conclusion

The combined Lab 7 work shows that clustering can recover meaningful structure in AIMO observations even when score columns are excluded from the clustering input.

K-means on the detailed 40-feature deviation representation found **three clusters**: one high-quality movement group and two lower-quality groups with different failure patterns. PCA showed that these groups differ both in overall deviation severity and in the type of deviation.

Hierarchical clustering on the 14-feature movement-quality representation found **two clusters**: one large general cluster and one smaller subgroup characterized mainly by **right-sided outward knee movement and heel rise**. PCA confirmed that this subgroup is separated mostly along a component associated with heel rise and outward-knee behaviour.

Overall, the sprint result is that both clustering methods were useful:

- **K-means** provided a compact partition with a richer multi-cluster structure
- **Hierarchical clustering** provided an interpretable split and a clear explanation of one distinct compensatory movement subgroup

Together, these methods give a stronger understanding of the AIMO data than either method alone.

## References to source notebooks

The combined report is based on the following notebooks:

- `reporting/A7_h.ipynb`
- `ml/models/research/A7/clustering_h.ipynb`
- `ml/models/research/A7/A7_cluster_kmeans.ipynb`